# Анализ CRM по сегментам бизнеса

Файл на сервере: `crm_last_version.csv`

**Сегментация по `X_AREA_RESP`:**

| X_AREA_RESP | Сегмент |
|-------------|--------|
| ДМСБ (микро) | МкБ |
| ДМСБ (малый) | МСБ |
| ДМСБ (средний) | МСБ |
| ДКБ | ККБ |

**Период:** 2023-2025 включительно  
**Дедупликация:** уникальные комбинации `X_INN` + `NUMBER_FP_SFP` + `IDENTIFICATION_DATE`

In [ ]:
import pandas as pd       # библиотека для работы с таблицами (DataFrame)
import numpy as np        # библиотека для числовых операций (используется pandas под капотом)
import matplotlib.pyplot as plt  # библиотека для построения графиков

import warnings
# Подавляем предупреждения (FutureWarning, DeprecationWarning и т.д.),
# чтобы вывод ячеек был чистым — только наши print/display.
# Код работает и без этой строки, но в выводе будут жёлтые предупреждения.
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)  # отображаем полные наименования в таблицах

# ── Пути к файлам ──
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"  # папка с исходными данными
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"             # выгрузка CRM с факторами проблемности
REF_FILE = f"{DATA_DIR}/ref_book.csv"                      # справочник ФП/СФП (номера, наименования, ЗО)

# ── Период анализа ──
DATE_FROM = pd.Timestamp("2023-01-01")  # начало периода (включительно)
DATE_TO   = pd.Timestamp("2025-12-31")  # конец периода (включительно)

# ── Маппинг X_AREA_RESP → укрупнённый сегмент бизнеса ──
# В CRM у каждой записи есть колонка X_AREA_RESP (зона ответственности).
# Мы объединяем 4 значения в 3 сегмента для анализа:
#   МкБ  = микробизнес
#   МСБ  = малый + средний бизнес (объединяем два подразделения ДМСБ)
#   ККБ  = крупный корпоративный бизнес
SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

# ── Маппинг X_AREA_RESP → ЗО для ФП/СФП (как в справочнике ref_book.csv) ──
# В справочнике колонка "ЗО для ФП/СФП" называется иначе, чем X_AREA_RESP в CRM.
# Этот маппинг нужен, чтобы при джойне CRM со справочником
# сопоставить правильные наименования факторов.
# Пример: в CRM "ДМСБ (микро)", а в справочнике та же ЗО называется "ДМ (микро)".
ZO_MAP = {
    "ДМСБ (микро)":   "ДМ (микро)",
    "ДМСБ (малый)":   "ДМСБ (малый)",
    "ДМСБ (средний)": "ДМСБ (средний)",
    "ДКБ":            "ДКБ",
}
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    """
    Приводит ИНН к единому строковому формату.

    Зачем это нужно:
    - Excel может сохранить ИНН как число, добавив ".0" (например "7707083893.0")
    - В разных выгрузках ИНН может быть с ведущими нулями ("007707083893") или без
    - Без нормализации pandas считает "7707083893", "7707083893.0" и "007707083893"
      разными значениями, хотя это один и тот же ИНН

    Логика:
    1. Убираем ".0" с конца (артефакт Excel)
    2. Убираем ведущие нули
    3. Дополняем нулями слева до 10 знаков (юрлицо) или 12 знаков (ИП)
    """
    if pd.isna(s):           # если значение пустое (NaN) — возвращаем None
        return None
    s = str(s).strip()       # приводим к строке, убираем пробелы по краям
    if s.endswith(".0"):     # "7707083893.0" → "7707083893"
        s = s[:-2]
    s = s.lstrip("0") or "0"  # "007707083893" → "7707083893"; сохраняем "0" если всё нули
    if len(s) <= 10:
        return s.zfill(10)   # дополняем до 10 знаков: "7707083893" → "7707083893"
    return s.zfill(12)       # дополняем до 12 знаков (для ИП с 12-значным ИНН)


print("Параметры заданы.")

## 1. Загрузка и подготовка данных

In [ ]:
# ── Загрузка CSV-файла ──
# sep=";"          — разделитель колонок (в российских выгрузках обычно точка с запятой)
# dtype=str — читаем все колонки как строки, чтобы избежать потери ведущих нулей в ИНН
# dtype=str        — читаем ВСЕ колонки как строки, чтобы pandas не превращал
#                    ИНН/КПП/ОГРН в числа (и не терял ведущие нули)
# low_memory=False — отключаем экономию памяти, чтобы pandas не угадывал типы по частям
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Загружено: {len(df):,} строк")

# ── Фильтр по источникам ──
if "VAL" in df.columns:
    before_src = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам ({', '.join(ALLOWED_SOURCES)}): "
          f"{len(df):,} строк (было {before_src:,})")
else:
    print(f"Колонка 'VAL' не найдена — фильтр по источникам не применён")

# ── Нормализация ИНН ──
# Применяем функцию normalize_inn к каждому значению колонки X_INN,
# результат записываем в новую колонку "inn"
df["inn"] = df["X_INN"].apply(normalize_inn)

# ── Преобразование даты ──
# Колонка IDENTIFICATION_DATE хранится как строка (например "15.03.2024").
# Преобразуем в формат даты pandas (Timestamp), чтобы можно было фильтровать по периоду.
# dayfirst=True — указываем, что формат ДД.ММ.ГГГГ (а не ММ/ДД/ГГГГ как в США)
# errors="coerce" — если строку не удаётся распарсить, ставим NaT (а не падаем с ошибкой)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

# ── Фильтр по периоду ──
# Оставляем только строки, где дата идентификации попадает в наш период 2023-2025.
# .copy() — создаём независимую копию DataFrame, чтобы дальнейшие изменения
# не вызывали предупреждений pandas (SettingWithCopyWarning)
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}: {len(df):,} строк")

# ── Переименование TYPE_FP ──
# В реальной выгрузке crm_last_version значения в колонке TYPE_FP записаны
# латиницей: "FP" и "SFP". Переименовываем в кириллицу для единообразия.
# Для синтетических данных (где уже "ФП"/"СФП") replace ничего не изменит.
df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})

# ── Маппинг сегментов ──
# Приводим X_AREA_RESP к строке и убираем пробелы по краям (на случай "ДКБ " с пробелом)
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
# Создаём новую колонку "segment", подставляя значение из словаря SEGMENT_MAP.
# Если X_AREA_RESP не найден в словаре — segment будет NaN.
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

# Проверяем, есть ли значения X_AREA_RESP, которые не попали в маппинг (неизвестные ЗО).
# Если есть — выводим их для диагностики, затем исключаем из анализа.
unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print(f"\nНемаппированные значения X_AREA_RESP (будут исключены):")
    print(unmapped.to_string())

# Оставляем только строки с известным сегментом
df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

# ── Дедупликация ──
# У одного ИНН может быть много записей в CRM. Нас интересуют уникальные комбинации:
#   ИНН + номер фактора + дата идентификации
# Если все три совпадают — это дубль (например, из-за повторной загрузки данных).
# Если у одного ИНН одинаковый номер фактора, но РАЗНЫЕ даты — это разные события
# (фактор возник повторно в другую дату), поэтому оба остаются.
df_before_dedup = df.copy()

before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + fp_num + TYPE_FP + DATE): {len(df):,} строк (удалено {before_dedup - len(df):,} дублей)")

# ── Колонка для группировки по месяцам ──
# Преобразуем дату в «период» (год-месяц), например 2024-03.
# Это нужно для построения помесячной динамики в секциях 6 и 7.
df["year_month"] = df["dt"].dt.to_period("M")

## 2. Проверка: ИНН в нескольких сегментах

In [ ]:
# ── Проверка: может ли один ИНН оказаться в нескольких сегментах? ──
# Такое возможно, если компания проходила по разным подразделениям
# (например, сначала в МкБ, потом перешла в МСБ).
# Это важно знать, потому что при подсчёте «уникальных ИНН по сегментам»
# такие ИНН будут посчитаны в каждом сегменте, и сумма превысит общее число.

# Для каждого ИНН считаем, сколько РАЗНЫХ сегментов у него есть
seg_per_inn = df.groupby("inn")["segment"].nunique()

# Отбираем ИНН, у которых больше одного сегмента
multi_seg = seg_per_inn[seg_per_inn > 1]

print("=" * 60)
print("ПРОВЕРКА: ИНН в нескольких сегментах")
print("=" * 60)
print(f"  Всего уникальных ИНН:              {len(seg_per_inn):,}")
print(f"  ИНН в одном сегменте:              {(seg_per_inn == 1).sum():,}")
print(f"  ИНН в нескольких сегментах:        {len(multi_seg):,}")

if len(multi_seg) > 0:
    # Выводим детали по каждому «мультисегментному» ИНН (до 15 штук)
    print(f"\n  Детали (до 15 примеров):")
    print("  " + "-" * 56)
    for inn in multi_seg.index[:15]:
        subset = df[df["inn"] == inn]             # все записи этого ИНН
        segs = subset["segment"].unique()         # в каких сегментах он встречается
        areas = subset["X_AREA_RESP"].unique()    # какие оригинальные значения X_AREA_RESP
        n = len(subset)                           # сколько всего записей
        print(f"  ИНН {inn}  ->  {n} записей")
        print(f"    X_AREA_RESP: {', '.join(areas)}")
        print(f"    Сегменты:    {', '.join(segs)}")
else:
    print("\n  Каждый ИНН принадлежит ровно одному сегменту.")

## 3. Уникальные ИНН по сегментам

In [ ]:
# ── Подсчёт уникальных ИНН по сегментам ──

# Общее число уникальных ИНН во всём DataFrame (без разделения на сегменты)
total_inn = df["inn"].nunique()

# Порядок сегментов для отображения (от малого к крупному)
seg_order = ["МкБ", "МСБ", "ККБ"]

# Для каждого сегмента считаем количество уникальных ИНН.
# .reindex(seg_order) — гарантирует порядок + подставляет 0, если сегмент пуст.
inn_by_seg = df.groupby("segment")["inn"].nunique().reindex(seg_order, fill_value=0)

print("=" * 50)
print("УНИКАЛЬНЫЕ ИНН")
print("=" * 50)
print(f"  Всего:  {total_inn:,}")
print(f"  " + "-" * 30)
for seg in seg_order:
    cnt = inn_by_seg.get(seg, 0)
    pct = cnt / total_inn * 100 if total_inn else 0  # доля от общего числа
    print(f"  {seg:<6s} {cnt:>8,}  ({pct:.1f}%)")

# Дублируем в виде таблицы для наглядности
tbl = pd.DataFrame({
    "Сегмент": seg_order,
    "Уникальных ИНН": [inn_by_seg.get(s, 0) for s in seg_order],
})
tbl.loc[len(tbl)] = ["ИТОГО", total_inn]  # строка-итого внизу
display(tbl.style.hide(axis="index"))      # отображаем без стандартного числового индекса

## 4. Общая статистика по ФП/СФП

In [ ]:
# Сколько всего уникальных номеров факторов встречается в данных за период
total_factors = df["NUMBER_FP_SFP"].nunique()

# Из них — сколько уникальных номеров встречаются как ФП
fp_factors = df[df["TYPE_FP"] == "ФП"]["NUMBER_FP_SFP"].nunique()

# И сколько — как СФП
sfp_factors = df[df["TYPE_FP"] == "СФП"]["NUMBER_FP_SFP"].nunique()

# Общее количество записей (событий)
total_events = len(df)
fp_events = (df["TYPE_FP"] == "ФП").sum()
sfp_events = (df["TYPE_FP"] == "СФП").sum()

print("=" * 60)
print("ОБЩАЯ СТАТИСТИКА ПО ФП/СФП")
print("=" * 60)
print(f"  Всего записей (событий):              {total_events:,}")
print(f"    из них ФП:                           {fp_events:,}")
print(f"    из них СФП:                          {sfp_events:,}")
print()
print(f"  Уникальных номеров факторов:           {total_factors}")
print(f"    встречаются как ФП:                  {fp_factors}")
print(f"    встречаются как СФП:                 {sfp_factors}")

In [ ]:
# Проверка: какие номера факторов встречаются и как ФП, и как СФП
fp_set = set(df[df["TYPE_FP"] == "ФП"]["NUMBER_FP_SFP"].dropna().unique())
sfp_set = set(df[df["TYPE_FP"] == "СФП"]["NUMBER_FP_SFP"].dropna().unique())
both = sorted(fp_set & sfp_set)

print("=" * 70)
print("ПРОВЕРКА: факторы, встречающиеся и как ФП, и как СФП")
print("=" * 70)
print(f"  Уникальных номеров только ФП:    {len(fp_set - sfp_set)}")
print(f"  Уникальных номеров только СФП:   {len(sfp_set - fp_set)}")
print(f"  Встречаются в обоих типах:       {len(both)}")
print(f"  Итого уникальных:                {len(fp_set | sfp_set)}")
print(f"  ({len(fp_set)} ФП + {len(sfp_set)} СФП - {len(both)} пересечение = {len(fp_set) + len(sfp_set) - len(both)})")
print()

if both:
    _ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
    _ref.columns = _ref.columns.str.strip()
    _name_map = _ref.drop_duplicates(subset=["№"]).set_index("№")["Наименование"].to_dict()

    print(f"  Номера факторов в пересечении ({len(both)} шт.):")
    print("  " + "-" * 66)
    for num in both:
        fp_cnt = (df[(df["NUMBER_FP_SFP"] == num) & (df["TYPE_FP"] == "ФП")]).shape[0]
        sfp_cnt = (df[(df["NUMBER_FP_SFP"] == num) & (df["TYPE_FP"] == "СФП")]).shape[0]
        name = _name_map.get(num, "— наименование не найдено в справочнике —")
        print(f"    {num:<10s}  ФП: {fp_cnt:>6,}  |  СФП: {sfp_cnt:>5,}")
        print(f"              {name}")
        print()
else:
    print("  Пересечений нет — все номера принадлежат только одному типу.")

## 5. Количество ФП/СФП по сегментам и среднее на ИНН

In [ ]:
# ── Кросс-таблица: количество записей по (сегмент × тип фактора) ──
fp_stats = df.groupby(["segment", "TYPE_FP"]).size().unstack(fill_value=0)
fp_stats = fp_stats.reindex(seg_order, fill_value=0)

for col in ["ФП", "СФП"]:
    if col not in fp_stats.columns:
        fp_stats[col] = 0

fp_stats = fp_stats[["ФП", "СФП"]]
fp_stats["Всего"] = fp_stats["ФП"] + fp_stats["СФП"]

print("=" * 60)
print("КОЛИЧЕСТВО ФП/СФП ПО СЕГМЕНТАМ")
print("=" * 60)

totals = fp_stats.sum()
fp_stats.loc["ИТОГО"] = totals
display(fp_stats)

# ── Среднее количество ФП/СФП на один ИНН по сегментам ──
# Для каждого сегмента: общее число событий / число уникальных ИНН
print()
print("=" * 60)
print("СРЕДНЕЕ КОЛИЧЕСТВО ФП/СФП НА ОДИН ИНН")
print("=" * 60)

avg_rows = []
for seg in seg_order:
    seg_df = df[df["segment"] == seg]
    n_inn = seg_df["inn"].nunique()
    if n_inn == 0:
        continue
    n_fp = (seg_df["TYPE_FP"] == "ФП").sum()
    n_sfp = (seg_df["TYPE_FP"] == "СФП").sum()
    n_total = len(seg_df)
    avg_rows.append({
        "Сегмент": seg,
        "Уник. ИНН": n_inn,
        "ФП": n_fp,
        "СФП": n_sfp,
        "Всего": n_total,
        "Ср. ФП/ИНН": round(n_fp / n_inn, 2),
        "Ср. СФП/ИНН": round(n_sfp / n_inn, 2),
        "Ср. всего/ИНН": round(n_total / n_inn, 2),
    })

# Строка ИТОГО
all_inn = df["inn"].nunique()
all_fp = (df["TYPE_FP"] == "ФП").sum()
all_sfp = (df["TYPE_FP"] == "СФП").sum()
all_total = len(df)
avg_rows.append({
    "Сегмент": "ИТОГО",
    "Уник. ИНН": all_inn,
    "ФП": all_fp,
    "СФП": all_sfp,
    "Всего": all_total,
    "Ср. ФП/ИНН": round(all_fp / all_inn, 2) if all_inn else 0,
    "Ср. СФП/ИНН": round(all_sfp / all_inn, 2) if all_inn else 0,
    "Ср. всего/ИНН": round(all_total / all_inn, 2) if all_inn else 0,
})

df_avg = pd.DataFrame(avg_rows)
display(df_avg.style.hide(axis="index"))

# ── Столбчатая диаграмма ──
fp_plot = fp_stats.drop("ИТОГО")
ax = fp_plot[["ФП", "СФП"]].plot(kind="bar", figsize=(8, 4), edgecolor="white")
ax.set_title("Количество ФП и СФП по сегментам", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Количество")
ax.set_xticklabels(seg_order, rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Топ-20 самых частых ФП

In [ ]:
# Считаем частоту каждого номера фактора среди записей с TYPE_FP = "ФП"
fp_top20 = (
    df[df["TYPE_FP"] == "ФП"]
    .groupby("NUMBER_FP_SFP").size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
fp_top20.columns = ["№ ФП", "Кол-во"]
total_fp = (df["TYPE_FP"] == "ФП").sum()
fp_top20["% от всех ФП"] = (fp_top20["Кол-во"] / total_fp * 100).round(2).astype(str) + "%"
fp_top20.index = range(1, len(fp_top20) + 1)

print("=" * 50)
print(f"ТОП-20 САМЫХ ЧАСТЫХ ФП  (всего ФП: {total_fp:,})")
print("=" * 50)
display(fp_top20)

## 7. Топ-20 самых частых СФП

In [ ]:
# Аналогично секции 6, но для СФП
sfp_top20 = (
    df[df["TYPE_FP"] == "СФП"]
    .groupby("NUMBER_FP_SFP").size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
sfp_top20.columns = ["№ СФП", "Кол-во"]
total_sfp = (df["TYPE_FP"] == "СФП").sum()
sfp_top20["% от всех СФП"] = (sfp_top20["Кол-во"] / total_sfp * 100).round(2).astype(str) + "%"
sfp_top20.index = range(1, len(sfp_top20) + 1)

print("=" * 50)
print(f"ТОП-20 САМЫХ ЧАСТЫХ СФП  (всего СФП: {total_sfp:,})")
print("=" * 50)
display(sfp_top20)

## 8. Топ-20 самых частых ФП по сегментам

In [ ]:
# Для каждого сегмента выводим топ-20 ФП по частоте
for seg in seg_order:
    seg_fp = df[(df["segment"] == seg) & (df["TYPE_FP"] == "ФП")]
    top = (
        seg_fp.groupby("NUMBER_FP_SFP").size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top.columns = ["№ ФП", "Кол-во"]
    seg_total = len(seg_fp)
    top["% от ФП сегмента"] = (top["Кол-во"] / seg_total * 100).round(2).astype(str) + "%"
    top.index = range(1, len(top) + 1)

    print("=" * 55)
    print(f"  {seg}  —  Топ-20 ФП  (всего ФП в сегменте: {seg_total:,})")
    print("=" * 55)
    display(top)
    print()

## 9. Топ-20 самых частых СФП по сегментам

In [ ]:
# Аналогично секции 8, но для СФП
for seg in seg_order:
    seg_sfp = df[(df["segment"] == seg) & (df["TYPE_FP"] == "СФП")]
    top = (
        seg_sfp.groupby("NUMBER_FP_SFP").size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top.columns = ["№ СФП", "Кол-во"]
    seg_total = len(seg_sfp)
    top["% от СФП сегмента"] = (top["Кол-во"] / seg_total * 100).round(2).astype(str) + "%" if seg_total > 0 else "—"
    top.index = range(1, len(top) + 1)

    print("=" * 55)
    print(f"  {seg}  —  Топ-20 СФП  (всего СФП в сегменте: {seg_total:,})")
    print("=" * 55)
    display(top)
    print()

## 10. Подключение справочника ФП/СФП (ref_book.csv)

In [ ]:
# ── Загрузка справочника ──
df_ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
df_ref.columns = df_ref.columns.str.strip()

# ── Маппинг X_AREA_RESP → ЗО (как в справочнике) ──
# Нужен для джойна: в CRM "ДМСБ (микро)", в справочнике "ДМ (микро)"
df["zo"] = df["X_AREA_RESP"].map(ZO_MAP)

# ── Джойн по двум ключам: номер фактора + ЗО ──
# Один и тот же номер (например "35.2") в разных ЗО имеет РАЗНОЕ наименование,
# поэтому джойн по двум ключам — обязателен.
df_merged = df.merge(
    df_ref[["№", "ЗО для ФП/СФП", "Наименование"]],
    left_on=["NUMBER_FP_SFP", "zo"],
    right_on=["№", "ЗО для ФП/СФП"],
    how="left",
)

no_match = df_merged["Наименование"].isna().sum()
total_rows = len(df_merged)
print(f"Джойн со справочником: {total_rows:,} строк, не найдено наименование: {no_match:,} ({no_match/total_rows*100:.1f}%)")


def build_top(data, n=5):
    """
    Топ-N номеров ФП/СФП по частоте с подтягиванием наименований.
    Считаем по NUMBER_FP_SFP (не по паре номер+наименование),
    потому что в МСБ один номер может иметь разные наименования из разных ЗО.
    """
    counts = data.groupby("NUMBER_FP_SFP").size().nlargest(n)
    rows = []
    for num, cnt in counts.items():
        names = data.loc[data["NUMBER_FP_SFP"] == num, "Наименование"].dropna().unique()
        rows.append({
            "№ ФП/СФП": num,
            "Наименование": " / ".join(sorted(names)) if len(names) > 0 else "",
            "Кол-во": cnt,
        })
    result = pd.DataFrame(rows)
    result.index = range(1, len(result) + 1)
    return result

## 11. Топ-5 самых частых ФП по сегментам (с наименованиями)

In [ ]:
full_fp_names = {}
for seg in seg_order:
    seg_data = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "ФП")]
    if len(seg_data) == 0:
        print(f"\n  {seg}: нет записей ФП")
        continue

    full_fp_names[seg] = build_top(seg_data, len(seg_data))
    zo_vals = sorted(seg_data["zo"].dropna().unique())
    zo_label = ", ".join(zo_vals)
    print("=" * 80)
    print(f"  {seg}  (ЗО: {zo_label})  —  Топ-5 самых частых ФП")
    print("=" * 80)
    display(build_top(seg_data, 5))
    print()

## 12. Топ-5 самых частых СФП по сегментам (с наименованиями)

In [ ]:
full_sfp_names = {}
for seg in seg_order:
    seg_data = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "СФП")]
    if len(seg_data) == 0:
        print(f"\n  {seg}: нет записей СФП")
        continue

    full_sfp_names[seg] = build_top(seg_data, len(seg_data))
    zo_vals = sorted(seg_data["zo"].dropna().unique())
    zo_label = ", ".join(zo_vals)
    print("=" * 80)
    print(f"  {seg}  (ЗО: {zo_label})  —  Топ-5 самых частых СФП")
    print("=" * 80)
    display(build_top(seg_data, 5))
    print()

## 13. Самые частые комбинации из 2 ФП по сегментам

Для каждого ИНН берём все уникальные номера ФП, генерируем все пары, считаем какие пары встречаются чаще всего.

In [ ]:
from itertools import combinations

combo_fp_by_seg = {}
for seg in seg_order:
    seg_fp = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "ФП")]

    inn_factors = seg_fp.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))

    pair_counts = {}
    for factors in inn_factors:
        if len(factors) >= 2:
            for pair in combinations(factors, 2):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    if not pair_counts:
        print(f"\n  {seg}: нет ИНН с 2+ различными ФП\n")
        continue

    all_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
    rows_all = []
    for (f1, f2), cnt in all_pairs:
        n1 = seg_fp.loc[seg_fp["NUMBER_FP_SFP"] == f1, "Наименование"].dropna().unique()
        n2 = seg_fp.loc[seg_fp["NUMBER_FP_SFP"] == f2, "Наименование"].dropna().unique()
        rows_all.append({
            "ФП 1": f1, "Наименование 1": " / ".join(sorted(n1)) if len(n1) else "",
            "ФП 2": f2, "Наименование 2": " / ".join(sorted(n2)) if len(n2) else "",
            "Кол-во ИНН": cnt,
        })
    full_result = pd.DataFrame(rows_all)
    full_result.index = range(1, len(full_result) + 1)
    combo_fp_by_seg[seg] = full_result

    zo_vals = sorted(seg_fp["zo"].dropna().unique())
    print("=" * 95)
    print(f"  {seg}  (ЗО: {', '.join(zo_vals)})  —  Топ комбинаций из 2 ФП")
    print("=" * 95)
    display(full_result.head(10))
    print()

## 14. Самые частые комбинации из 2 СФП по сегментам

In [ ]:
combo_sfp_by_seg = {}
for seg in seg_order:
    seg_sfp = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "СФП")]

    inn_factors = seg_sfp.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))

    pair_counts = {}
    for factors in inn_factors:
        if len(factors) >= 2:
            for pair in combinations(factors, 2):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    if not pair_counts:
        print(f"\n  {seg}: нет ИНН с 2+ различными СФП\n")
        continue

    all_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
    rows_all = []
    for (f1, f2), cnt in all_pairs:
        n1 = seg_sfp.loc[seg_sfp["NUMBER_FP_SFP"] == f1, "Наименование"].dropna().unique()
        n2 = seg_sfp.loc[seg_sfp["NUMBER_FP_SFP"] == f2, "Наименование"].dropna().unique()
        rows_all.append({
            "СФП 1": f1, "Наименование 1": " / ".join(sorted(n1)) if len(n1) else "",
            "СФП 2": f2, "Наименование 2": " / ".join(sorted(n2)) if len(n2) else "",
            "Кол-во ИНН": cnt,
        })
    full_result = pd.DataFrame(rows_all)
    full_result.index = range(1, len(full_result) + 1)
    combo_sfp_by_seg[seg] = full_result

    zo_vals = sorted(seg_sfp["zo"].dropna().unique())
    print("=" * 95)
    print(f"  {seg}  (ЗО: {', '.join(zo_vals)})  —  Топ комбинаций из 2 СФП")
    print("=" * 95)
    display(full_result.head(10))
    print()

## 15. Динамика ФП по месяцам (по сегментам)

In [ ]:
# ── Оставляем только строки с TYPE_FP = "ФП" (факторы проблемности) ──
# СФП (существенные факторы проблемности) анализируются отдельно в секции 7
df_fp = df[df["TYPE_FP"] == "ФП"].copy()

# ── Сводная таблица: кол-во ФП по (месяц × сегмент) ──
# groupby по year_month и segment, считаем количество строк,
# .unstack() разворачивает сегменты в колонки.
# Результат — таблица, где строки = месяцы, колонки = сегменты, значения = кол-во ФП.
pivot_fp = df_fp.groupby(["year_month", "segment"]).size().unstack(fill_value=0)

# Гарантируем порядок колонок (МкБ, МСБ, ККБ) и заполняем нулями пустые
pivot_fp = pivot_fp.reindex(columns=seg_order, fill_value=0)

# Преобразуем индекс из Period ("2024-03") в строку для корректного отображения на графике
pivot_fp.index = pivot_fp.index.astype(str)

# Таблица с точными значениями по месяцам
tbl_fp = pivot_fp.copy()
tbl_fp["Итого"] = tbl_fp[seg_order].sum(axis=1)
tbl_fp.index.name = "Месяц"

print("=" * 60)
print("ДИНАМИКА ФП ПО МЕСЯЦАМ (таблица)")
print("=" * 60)
display(tbl_fp)

# ── Линейный график ──
fig, ax = plt.subplots(figsize=(14, 5))           # создаём фигуру 14×5 дюймов
pivot_fp.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)  # линия с точками для каждого сегмента
ax.set_title("Количество выявленных ФП по месяцам (2023-2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество ФП")
ax.legend(title="Сегмент")               # легенда с заголовком «Сегмент»
ax.tick_params(axis="x", rotation=45)     # поворачиваем подписи месяцев на 45°, чтобы не слипались
plt.tight_layout()                        # автоподбор отступов
plt.show()

## 16. Динамика СФП по месяцам (по сегментам)

In [ ]:
# ── Аналогично секции 6, но для СФП (существенные факторы проблемности) ──
df_sfp = df[df["TYPE_FP"] == "СФП"].copy()

# Сводная таблица: кол-во СФП по (месяц × сегмент)
pivot_sfp = df_sfp.groupby(["year_month", "segment"]).size().unstack(fill_value=0)
pivot_sfp = pivot_sfp.reindex(columns=seg_order, fill_value=0)
pivot_sfp.index = pivot_sfp.index.astype(str)

# Таблица с точными значениями по месяцам
tbl_sfp = pivot_sfp.copy()
tbl_sfp["Итого"] = tbl_sfp[seg_order].sum(axis=1)
tbl_sfp.index.name = "Месяц"

print("=" * 60)
print("ДИНАМИКА СФП ПО МЕСЯЦАМ (таблица)")
print("=" * 60)
display(tbl_sfp)

# ── Линейный график ──
fig, ax = plt.subplots(figsize=(14, 5))
pivot_sfp.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)
ax.set_title("Количество выявленных СФП по месяцам (2023-2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество СФП")
ax.legend(title="Сегмент")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 17. Анализ сумм по договорам (APPROVED_SUM)

Для каждого уникального ИНН суммируем `APPROVED_SUM` по всем уникальным договорам (`AGREEMENT_NUM`).
Учитываются только рублёвые договоры (`CURCY_CD == "RUB"`).
Результат распределяется по 8 корзинам в разрезе по сегментам.

In [ ]:
# --- Фильтр: только RUB ---
df_rub = df[df["CURCY_CD"].astype(str).str.strip().str.upper() == "RUB"].copy()
print(f"Строк с CURCY_CD = RUB: {len(df_rub):,} из {len(df):,}")

# --- Приведение APPROVED_SUM к числу ---
df_rub["approved_sum"] = pd.to_numeric(
    df_rub["APPROVED_SUM"].astype(str).str.replace(",", ".").str.strip(),
    errors="coerce"
)
print(f"APPROVED_SUM заполнен: {df_rub['approved_sum'].notna().sum():,} ({df_rub['approved_sum'].notna().mean():.1%})")

# --- Дедупликация по AGREEMENT_NUM + INN ---
df_agr = (
    df_rub[df_rub["approved_sum"].notna()]
    .drop_duplicates(subset=["inn", "AGREEMENT_NUM"])
    [["inn", "segment", "AGREEMENT_NUM", "approved_sum"]]
    .copy()
)
print(f"Уникальных пар ИНН + договор: {len(df_agr):,}")

# --- Суммирование по ИНН ---
inn_sum = df_agr.groupby("inn").agg(
    total_sum=("approved_sum", "sum"),
    n_agreements=("AGREEMENT_NUM", "nunique"),
    segment=("segment", "first"),
).reset_index()
print(f"Уникальных ИНН с суммой: {len(inn_sum):,}")

# --- Корзины ---
BUCKET_LABELS = [
    "до 100 млн",
    "100–500 млн",
    "500 млн – 1 млрд",
    "1–2 млрд",
    "2–5 млрд",
    "5–10 млрд",
    "10–20 млрд",
    "свыше 20 млрд",
]
BUCKET_EDGES = [0, 1e8, 5e8, 1e9, 2e9, 5e9, 1e10, 2e10, float("inf")]

inn_sum["bucket"] = pd.cut(
    inn_sum["total_sum"], bins=BUCKET_EDGES, labels=BUCKET_LABELS,
    right=True, include_lowest=True,
)

# --- Кросс-таблица: корзина × сегмент (абсолютные) ---
cross_abs = pd.crosstab(
    inn_sum["bucket"], inn_sum["segment"],
    margins=True, margins_name="Итого"
).reindex(columns=seg_order + ["Итого"], fill_value=0)
cross_abs = cross_abs.reindex(BUCKET_LABELS + ["Итого"], fill_value=0)

print("\n" + "=" * 60)
print("РАСПРЕДЕЛЕНИЕ ИНН ПО КОРЗИНАМ APPROVED_SUM (абсолютные)")
print("=" * 60)
display(cross_abs)

# --- Кросс-таблица: корзина × сегмент (проценты внутри сегмента) ---
cross_pct = cross_abs.div(cross_abs.loc["Итого"]).mul(100).round(1)
cross_pct.loc["Итого"] = cross_abs.loc["Итого"]

print("\n" + "=" * 60)
print("РАСПРЕДЕЛЕНИЕ ИНН ПО КОРЗИНАМ APPROVED_SUM (% от сегмента)")
print("=" * 60)
display(cross_pct)

# --- Столбчатая диаграмма ---
plot_data = cross_abs.drop("Итого").drop(columns="Итого")
ax = plot_data.plot(kind="bar", figsize=(12, 5), edgecolor="white")
ax.set_title("Распределение ИНН по корзинам APPROVED_SUM (только RUB)", fontsize=13, fontweight="bold")
ax.set_xlabel("Корзина")
ax.set_ylabel("Количество ИНН")
ax.legend(title="Сегмент")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# --- Сводная статистика по сегментам ---
sum_stats = inn_sum.groupby("segment")["total_sum"].agg(
    ["count", "mean", "median", "min", "max"]
).reindex(seg_order)
sum_stats.columns = ["ИНН", "Среднее", "Медиана", "Мин", "Макс"]
for c in ["Среднее", "Медиана", "Мин", "Макс"]:
    sum_stats[c] = sum_stats[c].apply(lambda x: f"{x:,.0f}")

print("\n" + "=" * 60)
print("СТАТИСТИКА APPROVED_SUM ПО СЕГМЕНТАМ (RUB)")
print("=" * 60)
display(sum_stats)

## 18. Сохранение всех таблиц в Excel-приложение

In [ ]:
_t = lambda s: s[:31]
excel_path = f'{DATA_DIR}/Приложение к отчету "Анализ динамики ФП по сегментам бизнеса".xlsx'

with pd.ExcelWriter(excel_path, engine="openpyxl") as w:
    tbl.to_excel(w, sheet_name="Уникальные ИНН", index=False)

    fp_stats.to_excel(w, sheet_name="ФП-СФП по сегментам")
    df_avg.to_excel(w, sheet_name="Среднее ФП на ИНН", index=False)

    fp_all = (
        df[df["TYPE_FP"] == "ФП"]
        .groupby("NUMBER_FP_SFP").size()
        .sort_values(ascending=False)
        .reset_index()
    )
    fp_all.columns = ["№ ФП", "Кол-во"]
    total_fp = (df["TYPE_FP"] == "ФП").sum()
    fp_all["% от всех ФП"] = (fp_all["Кол-во"] / total_fp * 100).round(2).astype(str) + "%"
    fp_all.to_excel(w, sheet_name="Все ФП (общие)", index=False)

    sfp_all = (
        df[df["TYPE_FP"] == "СФП"]
        .groupby("NUMBER_FP_SFP").size()
        .sort_values(ascending=False)
        .reset_index()
    )
    sfp_all.columns = ["№ СФП", "Кол-во"]
    total_sfp = (df["TYPE_FP"] == "СФП").sum()
    sfp_all["% от всех СФП"] = (sfp_all["Кол-во"] / total_sfp * 100).round(2).astype(str) + "%"
    sfp_all.to_excel(w, sheet_name="Все СФП (общие)", index=False)

    for seg in seg_order:
        seg_fp_data = df[df["segment"] == seg]
        fp_t = seg_fp_data[seg_fp_data["TYPE_FP"] == "ФП"].groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).reset_index()
        fp_t.columns = ["№ ФП", "Кол-во"]
        fp_t.to_excel(w, sheet_name=_t(f"ФП ({seg})"), index=False)

        sfp_t = seg_fp_data[seg_fp_data["TYPE_FP"] == "СФП"].groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).reset_index()
        sfp_t.columns = ["№ СФП", "Кол-во"]
        sfp_t.to_excel(w, sheet_name=_t(f"СФП ({seg})"), index=False)

    for seg, df_fn in full_fp_names.items():
        df_fn.to_excel(w, sheet_name=_t(f"ФП с назв. ({seg})"), index=False)
    for seg, df_fn in full_sfp_names.items():
        df_fn.to_excel(w, sheet_name=_t(f"СФП с назв. ({seg})"), index=False)

    for seg, df_c in combo_fp_by_seg.items():
        df_c.to_excel(w, sheet_name=_t(f"Комбинации ФП ({seg})"), index=False)
    for seg, df_c in combo_sfp_by_seg.items():
        df_c.to_excel(w, sheet_name=_t(f"Комбинации СФП ({seg})"), index=False)

    tbl_fp.to_excel(w, sheet_name="Динамика ФП по месяцам")
    tbl_sfp.to_excel(w, sheet_name="Динамика СФП по месяцам")

    cross_abs.to_excel(w, sheet_name="APPROVED_SUM (абсолют.)")
    cross_pct.to_excel(w, sheet_name="APPROVED_SUM (%)")
    sum_stats.to_excel(w, sheet_name="APPROVED_SUM статистика")
    inn_sum.to_excel(w, sheet_name="APPROVED_SUM по ИНН", index=False)

print(f"Сохранено: {excel_path}")

## 19. Генерация Word-отчёта "Анализ динамики ФП по сегментам бизнеса"

In [ ]:
import os, tempfile
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import nsdecls, qn
from docx.oxml import parse_xml, OxmlElement

import matplotlib
matplotlib.use("Agg")

_OUT = f'{DATA_DIR}/Анализ динамики ФП по сегментам бизнеса.docx'
_TMP = tempfile.mkdtemp()
_HDR = "1F4E79"
_ALT = "D6E4F0"

def _shade(cell, color):
    cell._tc.get_or_add_tcPr().append(parse_xml(f'<w:shd {nsdecls("w")} w:fill="{color}"/>'))

def _cw(cell, text, sz=9, bold=False, align=None, white=False):
    cell.text = ""
    p = cell.paragraphs[0]
    if align: p.alignment = align
    r = p.add_run(str(text))
    r.font.size = Pt(sz); r.font.bold = bold; r.font.name = "Calibri"
    if white: r.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)

def _set_widths(table, widths_cm):
    tbl = table._tbl
    tblPr = tbl.tblPr if tbl.tblPr is not None else OxmlElement("w:tblPr")
    layout = OxmlElement("w:tblLayout")
    layout.set(qn("w:type"), "fixed"); tblPr.append(layout)
    grid = tbl.find(qn("w:tblGrid"))
    if grid is None:
        grid = OxmlElement("w:tblGrid"); tbl.insert(1, grid)
    else:
        for ch in list(grid): grid.remove(ch)
    for w in widths_cm:
        gc = OxmlElement("w:gridCol"); gc.set(qn("w:w"), str(int(w * 567))); grid.append(gc)
    for row in table.rows:
        for i, w in enumerate(widths_cm):
            if i < len(row.cells):
                tc = row.cells[i]._tc; tcPr = tc.get_or_add_tcPr()
                tw = OxmlElement("w:tcW"); tw.set(qn("w:w"), str(int(w * 567))); tw.set(qn("w:type"), "dxa")
                ex = tcPr.find(qn("w:tcW"))
                if ex is not None: tcPr.remove(ex)
                tcPr.insert(0, tw)

def _mktable(doc, headers, rows, font_sz=9, num_cols=None, col_widths=None):
    t = doc.add_table(rows=1+len(rows), cols=len(headers))
    t.style = "Table Grid"; t.alignment = WD_TABLE_ALIGNMENT.CENTER
    if num_cols is None: num_cols = set()
    for i, h in enumerate(headers):
        _cw(t.rows[0].cells[i], h, font_sz, bold=True, align=WD_ALIGN_PARAGRAPH.CENTER, white=True)
        _shade(t.rows[0].cells[i], _HDR)
    for ri, row in enumerate(rows):
        for ci, val in enumerate(row):
            al = WD_ALIGN_PARAGRAPH.RIGHT if ci in num_cols else None
            _cw(t.rows[ri+1].cells[ci], val, font_sz, align=al)
        if ri % 2 == 1:
            for ci in range(len(headers)): _shade(t.rows[ri+1].cells[ci], _ALT)
    if col_widths: _set_widths(t, col_widths)
    doc.add_paragraph()
    return t

def _add_toc(doc):
    p = doc.add_paragraph()
    r = p.add_run(); fc = OxmlElement("w:fldChar"); fc.set(qn("w:fldCharType"), "begin"); r._r.append(fc)
    r2 = p.add_run(); ins = OxmlElement("w:instrText"); ins.set(qn("xml:space"), "preserve")
    ins.text = ' TOC \\o "1-2" \\h \\z \\u '; r2._r.append(ins)
    r3 = p.add_run(); fs = OxmlElement("w:fldChar"); fs.set(qn("w:fldCharType"), "separate"); r3._r.append(fs)
    r4 = p.add_run("(Оглавление — нажмите Ctrl+A, затем F9 для обновления)")
    r4.font.size = Pt(10); r4.font.italic = True; r4.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
    r5 = p.add_run(); fe = OxmlElement("w:fldChar"); fe.set(qn("w:fldCharType"), "end"); r5._r.append(fe)

def _heading(doc, text, level=2):
    h = doc.add_heading(text, level=level)
    for run in h.runs: run.font.color.rgb = RGBColor(0, 0, 0)

def _para(doc, text):
    doc.add_paragraph(text)

def _add_chart(doc, path, w=15):
    doc.add_picture(path, width=Cm(w))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; doc.add_paragraph()

def _fmt(n):
    return f"{n:,}".replace(",", " ")

print("Утилиты для генерации .docx загружены.")

In [ ]:
doc = Document()
for sec in doc.sections:
    sec.top_margin = Cm(2); sec.bottom_margin = Cm(1.5)
    sec.left_margin = Cm(2); sec.right_margin = Cm(1.5)
style = doc.styles["Normal"]
style.font.name = "Calibri"; style.font.size = Pt(11); style.font.color.rgb = RGBColor(0, 0, 0)
for hn in ["Heading 1", "Heading 2"]:
    if hn in doc.styles: doc.styles[hn].font.color.rgb = RGBColor(0, 0, 0)

# --- Титульная страница ---
p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("Отчёт по результатам анализа CRM\nпо сегментам бизнеса")
r.font.size = Pt(20); r.font.bold = True; r.font.color.rgb = RGBColor(0, 0, 0)
p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run(f"Период анализа: {DATE_FROM:%Y}–{DATE_TO:%Y}")
r.font.size = Pt(14); r.font.color.rgb = RGBColor(0, 0, 0)
p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("Сегменты: МкБ (микро) · МСБ (малый + средний) · ККБ (ДКБ)")
r.font.size = Pt(11); r.font.italic = True
doc.add_page_break()

_heading(doc, "Оглавление", level=1); _add_toc(doc); doc.add_page_break()

# ═══ 1. Уникальные ИНН ═══
_heading(doc, "1. Уникальные ИНН по сегментам")
_total_inn = df["inn"].nunique()
_inn_seg = df.groupby("segment")["inn"].nunique().reindex(seg_order, fill_value=0)
_para(doc, f"Всего уникальных ИНН за анализируемый период: {_fmt(_total_inn)}")
rows_1 = [(s, _fmt(_inn_seg[s]), f"{_inn_seg[s]/_total_inn*100:.1f}%") for s in seg_order]
rows_1.append(("ИТОГО", _fmt(_total_inn), "100%"))
_mktable(doc, ["Сегмент", "Уникальных ИНН", "Доля"], rows_1, num_cols={1}, col_widths=[4, 5, 3])

# ═══ 1.1. Обоснование ключа дедупликации ═══
_heading(doc, "1.1. Обоснование ключа дедупликации")
_para(doc,
    "В выгрузке CRM каждая строка — карточка ФП/СФП, привязанная к конкретному "
    "кредитному договору (ROW_ID — уникальный идентификатор карточки). При возникновении "
    "ФП у компании карточка автоматически тиражируется на все её действующие договоры. "
    "Поэтому при подсчёте уникальных событий ФП необходимо выбрать правильный ключ дедупликации.")

_KEY_A = ["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]
_KEY_B = ["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE", "ROW_ID"]
_n_raw = len(df_before_dedup)
_n_key_a = df_before_dedup.drop_duplicates(subset=_KEY_A).shape[0]
_n_key_b = df_before_dedup.drop_duplicates(subset=_KEY_B).shape[0]
_delta = _n_key_b - _n_key_a
_n_multi = (df_before_dedup.groupby(_KEY_A)["ROW_ID"].nunique() > 1).sum()

_para(doc, "Было проведено сравнение двух ключей дедупликации:")
_mktable(doc,
    ["Ключ дедупликации", "Уникальных записей", "Δ к текущему"],
    [
        ("A: INN + NUMBER_FP_SFP + DATE (текущий)", _fmt(_n_key_a), "—"),
        ("B: A + ROW_ID (расширенный)", _fmt(_n_key_b), f"+{_fmt(_delta)} (+{_delta/_n_key_a*100:.1f}%)"),
    ],
    num_cols={1}, col_widths=[8, 3.5, 4])

_para(doc,
    f"Из {_fmt(_n_key_a)} уникальных групп (INN + ФП + дата) {_fmt(_n_multi)} групп "
    f"({_n_multi/_n_key_a*100:.1f}%) содержат более одного ROW_ID. "
    "Анализ примеров показал, что множественные ROW_ID внутри одной группы "
    "представляют собой копии одной и той же карточки ФП, растиражированные "
    "на разные договоры компании, а не независимые события выявления фактора.")

_para(doc,
    "Вывод: ключ A (INN + NUMBER_FP_SFP + IDENTIFICATION_DATE) является корректным "
    "для подсчёта уникальных событий ФП. Включение ROW_ID в ключ привело бы "
    f"к завышению числа событий на {_delta/_n_key_a*100:.1f}% за счёт учёта "
    "технических копий карточек по договорам.")

# ═══ 2. Общая статистика ═══
_heading(doc, "2. Общая статистика по ФП/СФП")
_te = len(df); _fe = (df["TYPE_FP"]=="ФП").sum(); _se = (df["TYPE_FP"]=="СФП").sum()
_uf = df["NUMBER_FP_SFP"].nunique()
_ff = df[df["TYPE_FP"]=="ФП"]["NUMBER_FP_SFP"].nunique()
_sf = df[df["TYPE_FP"]=="СФП"]["NUMBER_FP_SFP"].nunique()
rows_2 = [
    ("Всего записей (событий)", _fmt(_te)),
    ("   из них ФП", _fmt(_fe)), ("   из них СФП", _fmt(_se)),
    ("Уникальных номеров факторов", str(_uf)),
    ("   встречаются как ФП", str(_ff)), ("   встречаются как СФП", str(_sf)),
]
_mktable(doc, ["Показатель", "Значение"], rows_2, num_cols={1}, col_widths=[10, 4])

_fp_both = len(set(df[df["TYPE_FP"]=="ФП"]["NUMBER_FP_SFP"].dropna().unique()) & set(df[df["TYPE_FP"]=="СФП"]["NUMBER_FP_SFP"].dropna().unique()))
_para(doc,
    f"Факторы проблемности (ФП) составляют подавляющее большинство событий: "
    f"{_fmt(_fe)} из {_fmt(_te)} ({_fe/_te*100:.1f}%). "
    f"Существенные факторы проблемности (СФП) — всего {_fmt(_se)} событий ({_se/_te*100:.1f}%). "
    f"Из {_uf} уникальных номеров факторов {_ff} встречаются как ФП, {_sf} — как СФП"
    f"{f', {_fp_both} — в обоих типах' if _fp_both > 0 else ''}.")

# ═══ 3. ФП/СФП по сегментам + среднее ═══
_heading(doc, "3. Количество ФП/СФП по сегментам и среднее на ИНН")
rows_3a = []
for s in seg_order:
    sd = df[df["segment"] == s]
    nfp = (sd["TYPE_FP"]=="ФП").sum(); nsfp = (sd["TYPE_FP"]=="СФП").sum()
    rows_3a.append((s, _fmt(nfp), _fmt(nsfp), _fmt(nfp+nsfp)))
rows_3a.append(("ИТОГО", _fmt(_fe), _fmt(_se), _fmt(_te)))
_mktable(doc, ["Сегмент","ФП","СФП","Всего"], rows_3a, num_cols={1,2,3}, col_widths=[4,3.5,3.5,3.5])

rows_3b = []
for s in seg_order:
    sd = df[df["segment"] == s]; ni = sd["inn"].nunique()
    nfp = (sd["TYPE_FP"]=="ФП").sum(); nsfp = (sd["TYPE_FP"]=="СФП").sum(); nt = len(sd)
    rows_3b.append((s, _fmt(ni), _fmt(nfp), _fmt(nsfp), _fmt(nt),
                    f"{nfp/ni:.2f}" if ni else "—", f"{nsfp/ni:.2f}" if ni else "—",
                    f"{nt/ni:.2f}" if ni else "—"))
rows_3b.append(("ИТОГО", _fmt(_total_inn), _fmt(_fe), _fmt(_se), _fmt(_te),
                f"{_fe/_total_inn:.2f}", f"{_se/_total_inn:.2f}", f"{_te/_total_inn:.2f}"))
_mktable(doc, ["Сегмент","Уник. ИНН","ФП","СФП","Всего","Ср. ФП/ИНН","Ср. СФП/ИНН","Ср. всего/ИНН"],
         rows_3b, font_sz=8, num_cols={1,2,3,4,5,6,7}, col_widths=[2.5,2.2,2.2,1.8,2.2,2.2,2.2,2.2])

fig, ax = plt.subplots(figsize=(8, 4.5))
_bar_data = df.groupby(["segment","TYPE_FP"]).size().unstack(fill_value=0).reindex(seg_order, fill_value=0)
for c in ["ФП","СФП"]:
    if c not in _bar_data.columns: _bar_data[c] = 0
x = range(len(seg_order)); w_ = 0.35
b1 = ax.bar([i-w_/2 for i in x], _bar_data["ФП"], w_, label="ФП", color="#4472C4", edgecolor="white")
b2 = ax.bar([i+w_/2 for i in x], _bar_data["СФП"], w_, label="СФП", color="#ED7D31", edgecolor="white")
ax.set_title("Количество ФП и СФП по сегментам", fontsize=13, fontweight="bold")
ax.set_xticks(list(x)); ax.set_xticklabels(seg_order); ax.set_ylabel("Количество")
ax.bar_label(b1, fmt="%d", fontsize=8); ax.bar_label(b2, fmt="%d", fontsize=8); ax.legend()
plt.tight_layout()
_p1 = os.path.join(_TMP, "bar_seg.png"); fig.savefig(_p1, dpi=180); plt.close(fig)
_add_chart(doc, _p1)

_seg_stats = {}
for s in seg_order:
    sd = df[df["segment"] == s]; ni = sd["inn"].nunique()
    _seg_stats[s] = {"inn": ni, "total": len(sd), "avg": len(sd)/ni if ni else 0}
_max_seg = max(seg_order, key=lambda s: _seg_stats[s]["avg"])
_min_seg = min(seg_order, key=lambda s: _seg_stats[s]["avg"])
_para(doc,
    f"МкБ — самый массовый сегмент ({_fmt(_seg_stats['МкБ']['inn'])} ИНН, "
    f"{_seg_stats['МкБ']['inn']/_total_inn*100:.0f}% всех компаний), "
    f"но с наименьшей средней нагрузкой ({_seg_stats[_min_seg]['avg']:.2f} событий на ИНН). "
    f"Наибольшая нагрузка — в сегменте {_max_seg} "
    f"({_seg_stats[_max_seg]['avg']:.2f} событий на ИНН), "
    f"что в {_seg_stats[_max_seg]['avg']/_seg_stats[_min_seg]['avg']:.1f} раза выше, чем в {_min_seg}. "
    f"ККБ занимает промежуточное положение ({_seg_stats['ККБ']['avg']:.2f} событий на ИНН).")

# ═══ 4. Топ-20 ФП ═══
_heading(doc, "4. Топ-20 самых частых ФП")
_t20fp = df[df["TYPE_FP"]=="ФП"].groupby("NUMBER_FP_SFP").size().nlargest(20)
rows_4 = [(str(i+1), n, _fmt(c), f"{c/_fe*100:.2f}%") for i,(n,c) in enumerate(_t20fp.items())]
_mktable(doc, ["#","№ ФП","Кол-во","% от всех ФП"], rows_4, num_cols={0,2,3}, col_widths=[1,2.5,2.5,3])

_top5_fp_sum = _t20fp.head(5).sum()
_top10_fp_sum = _t20fp.head(10).sum()
_para(doc,
    f"Лидер — ФП {_t20fp.index[0]} ({_fmt(_t20fp.iloc[0])} событий, {_t20fp.iloc[0]/_fe*100:.1f}%). "
    f"Топ-5 ФП покрывают {_top5_fp_sum/_fe*100:.1f}% всех событий ФП, "
    f"топ-10 — {_top10_fp_sum/_fe*100:.1f}%. "
    "Высокая концентрация указывает на то, что относительно небольшое число типов факторов "
    "формирует основную массу событий.")

# ═══ 5. Топ-20 СФП ═══
_heading(doc, "5. Топ-20 самых частых СФП")
_t20sfp = df[df["TYPE_FP"]=="СФП"].groupby("NUMBER_FP_SFP").size().nlargest(20)
rows_5 = [(str(i+1), n, _fmt(c), f"{c/_se*100:.2f}%") for i,(n,c) in enumerate(_t20sfp.items())]
_mktable(doc, ["#","№ СФП","Кол-во","% от всех СФП"], rows_5, num_cols={0,2,3}, col_widths=[1,2.5,2.5,3])

_top3_sfp_sum = _t20sfp.head(3).sum()
_para(doc,
    f"Среди СФП наблюдается сверхконцентрация: лидер — СФП {_t20sfp.index[0]} "
    f"({_fmt(_t20sfp.iloc[0])} событий, {_t20sfp.iloc[0]/_se*100:.1f}%). "
    f"Топ-3 СФП покрывают {_top3_sfp_sum/_se*100:.1f}% всех событий СФП. "
    "Это означает, что большая часть существенных факторов проблемности "
    "связана с ограниченным числом причин.")

# ═══ 6. Топ-20 ФП по сегментам ═══
_heading(doc, "6. Топ-20 ФП по сегментам")
for seg in seg_order:
    sd = df[(df["segment"]==seg) & (df["TYPE_FP"]=="ФП")]
    stot = len(sd)
    t20 = sd.groupby("NUMBER_FP_SFP").size().nlargest(20)
    doc.add_paragraph(f"{seg} — Топ-20 ФП (всего: {_fmt(stot)})", style="List Bullet").runs[0].bold = True
    rows_ = [(str(i+1), n, _fmt(c), f"{c/stot*100:.2f}%") for i,(n,c) in enumerate(t20.items())]
    _mktable(doc, ["#","№ ФП","Кол-во","% от ФП сегмента"], rows_, num_cols={0,2,3}, col_widths=[1,2.5,2.5,3])

_seg_leaders = {}
for s in seg_order:
    _seg_leaders[s] = df[(df["segment"]==s) & (df["TYPE_FP"]=="ФП")].groupby("NUMBER_FP_SFP").size().nlargest(3).index.tolist()
_para(doc,
    "Межсегментные различия: лидирующие ФП существенно различаются между сегментами. "
    f"В МкБ доминируют ФП {', '.join(_seg_leaders.get('МкБ', [])[:2])}; "
    f"в МСБ — ФП {', '.join(_seg_leaders.get('МСБ', [])[:2])}; "
    f"в ККБ — ФП {', '.join(_seg_leaders.get('ККБ', [])[:2])}. "
    "Это подтверждает необходимость посегментного анализа: единый набор «топовых» факторов "
    "не может адекватно описать ситуацию во всех сегментах.")

# ═══ 7. Топ-20 СФП по сегментам ═══
_heading(doc, "7. Топ-20 СФП по сегментам")
for seg in seg_order:
    sd = df[(df["segment"]==seg) & (df["TYPE_FP"]=="СФП")]
    stot = len(sd)
    t20 = sd.groupby("NUMBER_FP_SFP").size().nlargest(20)
    doc.add_paragraph(f"{seg} — Топ-20 СФП (всего: {_fmt(stot)})", style="List Bullet").runs[0].bold = True
    rows_ = [(str(i+1), n, _fmt(c), f"{c/stot*100:.2f}%" if stot else "—") for i,(n,c) in enumerate(t20.items())]
    _mktable(doc, ["#","№ СФП","Кол-во","% от СФП сегмента"], rows_, num_cols={0,2,3}, col_widths=[1,2.5,2.5,3])

_sfp_leaders = {}
for s in seg_order:
    _sd = df[(df["segment"]==s) & (df["TYPE_FP"]=="СФП")]
    if len(_sd) > 0:
        _top1 = _sd.groupby("NUMBER_FP_SFP").size().nlargest(1)
        _sfp_leaders[s] = (_top1.index[0], _top1.iloc[0], _top1.iloc[0]/len(_sd)*100)
_para(doc,
    "Структура СФП по сегментам неоднородна. "
    + (f"В МСБ наблюдается сверхконцентрация: СФП {_sfp_leaders['МСБ'][0]} покрывает "
       f"{_sfp_leaders['МСБ'][2]:.0f}% всех СФП сегмента. " if "МСБ" in _sfp_leaders else "")
    + (f"В ККБ лидирует СФП {_sfp_leaders['ККБ'][0]} ({_sfp_leaders['ККБ'][2]:.0f}%). " if "ККБ" in _sfp_leaders else "")
    + (f"В МкБ распределение более равномерное: лидер — СФП {_sfp_leaders['МкБ'][0]} ({_sfp_leaders['МкБ'][2]:.0f}%)." if "МкБ" in _sfp_leaders else ""))

# ═══ 8. Топ-5 ФП с наименованиями ═══
_heading(doc, "8. Топ-5 ФП по сегментам с наименованиями")
for seg in seg_order:
    sd = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="ФП")]
    if len(sd) == 0: continue
    t5 = build_top(sd, 5)
    doc.add_paragraph(f"{seg}", style="List Bullet").runs[0].bold = True
    rows_ = [(str(i+1), t5.iloc[i]["№ ФП/СФП"], t5.iloc[i]["Наименование"], _fmt(t5.iloc[i]["Кол-во"])) for i in range(len(t5))]
    _mktable(doc, ["#","№ ФП/СФП","Наименование","Кол-во"], rows_, font_sz=8, num_cols={0,3}, col_widths=[0.8,2,12,2])

_para(doc,
    "Наименования подтверждают различную природу факторов по сегментам: "
    "в МкБ преобладают нарушения обязательств по поддержанию оборотов и непредставление документов; "
    "в МСБ — изменение финансовых показателей и ковенантные нарушения; "
    "в ККБ — сигналы внутренней рейтинговой системы и критериальные индикаторы.")

# ═══ 9. Топ-5 СФП с наименованиями ═══
_heading(doc, "9. Топ-5 СФП по сегментам с наименованиями")
for seg in seg_order:
    sd = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="СФП")]
    if len(sd) == 0: continue
    t5 = build_top(sd, 5)
    doc.add_paragraph(f"{seg}", style="List Bullet").runs[0].bold = True
    rows_ = [(str(i+1), t5.iloc[i]["№ ФП/СФП"], t5.iloc[i]["Наименование"], _fmt(t5.iloc[i]["Кол-во"])) for i in range(len(t5))]
    _mktable(doc, ["#","№ ФП/СФП","Наименование","Кол-во"], rows_, font_sz=8, num_cols={0,3}, col_widths=[0.8,2,12,2])

_para(doc,
    "Среди СФП в МкБ лидируют факторы, связанные с просроченной задолженностью и банкротством; "
    "в МСБ — прекращение действия разрешений и процессы ликвидации; "
    "в ККБ — возбуждение уголовных дел и ликвидационные процедуры.")

# ═══ 10. Комбинации из 2 ФП ═══
_heading(doc, "10. Самые частые комбинации из 2 ФП по сегментам")
for seg in seg_order:
    seg_fp = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="ФП")]
    inn_f = seg_fp.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))
    pc = {}
    for facs in inn_f:
        if len(facs) >= 2:
            for pr in combinations(facs, 2): pc[pr] = pc.get(pr, 0) + 1
    if not pc:
        doc.add_paragraph(f"{seg}: нет ИНН с 2+ различными ФП"); continue
    top_ = sorted(pc.items(), key=lambda x: -x[1])[:10]
    nmap = {}
    for (f1, f2), cnt in top_:
        for fn in (f1, f2):
            if fn not in nmap:
                ns = seg_fp.loc[seg_fp["NUMBER_FP_SFP"]==fn, "Наименование"].dropna().unique()
                nmap[fn] = " / ".join(sorted(ns)) if len(ns) else ""
    doc.add_paragraph(f"{seg}", style="List Bullet").runs[0].bold = True
    rows_ = [(str(i+1), f1, nmap.get(f1,""), f2, nmap.get(f2,""), _fmt(cnt))
             for i,((f1,f2),cnt) in enumerate(top_)]
    _mktable(doc, ["#","ФП 1","Наименование 1","ФП 2","Наименование 2","Кол-во ИНН"],
             rows_, font_sz=7, num_cols={0,5}, col_widths=[0.7,1.3,5.8,1.3,5.8,1.8])

_para(doc,
    "Анализ парных комбинаций ФП позволяет выявить устойчивые «связки» факторов, "
    "которые чаще всего возникают у одних и тех же компаний. Наиболее частые пары различаются "
    "по сегментам, что отражает специфику кредитного портфеля каждого направления.")

# ═══ 11. Комбинации из 2 СФП ═══
_heading(doc, "11. Самые частые комбинации из 2 СФП по сегментам")
for seg in seg_order:
    seg_sfp_d = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="СФП")]
    inn_f = seg_sfp_d.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))
    pc = {}
    for facs in inn_f:
        if len(facs) >= 2:
            for pr in combinations(facs, 2): pc[pr] = pc.get(pr, 0) + 1
    if not pc:
        doc.add_paragraph(f"{seg}: нет ИНН с 2+ различными СФП"); continue
    top_ = sorted(pc.items(), key=lambda x: -x[1])[:10]
    nmap = {}
    for (f1, f2), cnt in top_:
        for fn in (f1, f2):
            if fn not in nmap:
                ns = seg_sfp_d.loc[seg_sfp_d["NUMBER_FP_SFP"]==fn, "Наименование"].dropna().unique()
                nmap[fn] = " / ".join(sorted(ns)) if len(ns) else ""
    doc.add_paragraph(f"{seg}", style="List Bullet").runs[0].bold = True
    rows_ = [(str(i+1), f1, nmap.get(f1,""), f2, nmap.get(f2,""), _fmt(cnt))
             for i,((f1,f2),cnt) in enumerate(top_)]
    _mktable(doc, ["#","СФП 1","Наименование 1","СФП 2","Наименование 2","Кол-во ИНН"],
             rows_, font_sz=7, num_cols={0,5}, col_widths=[0.7,1.3,5.8,1.3,5.8,1.8])

_para(doc,
    "Комбинации СФП встречаются значительно реже, чем комбинации ФП, что объясняется "
    "малым общим числом СФП-событий. Наличие парных связок СФП у компании "
    "может свидетельствовать о системных проблемах заёмщика.")

# ═══ 12. Динамика ФП ═══
_heading(doc, "12. Динамика ФП по месяцам (по сегментам)")
_pvt_fp = df[df["TYPE_FP"]=="ФП"].groupby(["year_month","segment"]).size().unstack(fill_value=0).reindex(columns=seg_order, fill_value=0)
_pvt_fp.index = _pvt_fp.index.astype(str)
_pvt_fp["Итого"] = _pvt_fp[seg_order].sum(axis=1)
dyn_rows_fp = [(m, *[_fmt(int(_pvt_fp.loc[m, c])) for c in seg_order + ["Итого"]]) for m in _pvt_fp.index]
_mktable(doc, ["Месяц","МкБ","МСБ","ККБ","Итого"], dyn_rows_fp, font_sz=8, num_cols={1,2,3,4},
         col_widths=[3,2.5,2.5,2.5,2.5])

fig, ax = plt.subplots(figsize=(14, 5))
colors = {"МкБ": "#4472C4", "МСБ": "#ED7D31", "ККБ": "#70AD47"}
for seg in seg_order:
    vals = [int(_pvt_fp.loc[m, seg]) for m in _pvt_fp.index]
    ax.plot(_pvt_fp.index, vals, marker="o", markersize=3, label=seg, color=colors[seg], linewidth=1.5)
ax.set_title("Динамика ФП по месяцам", fontsize=13, fontweight="bold")
ax.set_ylabel("Количество ФП"); ax.legend(); ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
_p2 = os.path.join(_TMP, "dyn_fp.png"); fig.savefig(_p2, dpi=180); plt.close(fig)
_add_chart(doc, _p2)

_fp_totals = _pvt_fp["Итого"]
_fp_peak_month = _fp_totals.idxmax()
_fp_peak_val = int(_fp_totals.max())
_fp_min_month = _fp_totals.idxmin()
_fp_min_val = int(_fp_totals.min())
_para(doc,
    f"Динамика ФП демонстрирует выраженную сезонность. "
    f"Максимальное число событий зафиксировано в {_fp_peak_month} ({_fmt(_fp_peak_val)}), "
    f"минимальное — в {_fp_min_month} ({_fmt(_fp_min_val)}). "
    "Характерные пики приходятся на апрель–май и август каждого года, "
    "что может быть связано с плановыми проверками и пересмотром кредитных портфелей. "
    "Во всех сегментах динамика синхронна — пики и спады происходят одновременно.")

# ═══ 13. Динамика СФП ═══
_heading(doc, "13. Динамика СФП по месяцам (по сегментам)")
_pvt_sfp = df[df["TYPE_FP"]=="СФП"].groupby(["year_month","segment"]).size().unstack(fill_value=0).reindex(columns=seg_order, fill_value=0)
_pvt_sfp.index = _pvt_sfp.index.astype(str)
_pvt_sfp["Итого"] = _pvt_sfp[seg_order].sum(axis=1)
dyn_rows_sfp = [(m, *[_fmt(int(_pvt_sfp.loc[m, c])) for c in seg_order + ["Итого"]]) for m in _pvt_sfp.index]
_mktable(doc, ["Месяц","МкБ","МСБ","ККБ","Итого"], dyn_rows_sfp, font_sz=8, num_cols={1,2,3,4},
         col_widths=[3,2.5,2.5,2.5,2.5])

fig, ax = plt.subplots(figsize=(14, 5))
for seg in seg_order:
    vals = [int(_pvt_sfp.loc[m, seg]) for m in _pvt_sfp.index]
    ax.plot(_pvt_sfp.index, vals, marker="o", markersize=3, label=seg, color=colors[seg], linewidth=1.5)
ax.set_title("Динамика СФП по месяцам", fontsize=13, fontweight="bold")
ax.set_ylabel("Количество СФП"); ax.legend(); ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
_p3 = os.path.join(_TMP, "dyn_sfp.png"); fig.savefig(_p3, dpi=180); plt.close(fig)
_add_chart(doc, _p3)

_sfp_totals = _pvt_sfp["Итого"]
_sfp_peak_month = _sfp_totals.idxmax()
_sfp_peak_val = int(_sfp_totals.max())
_para(doc,
    f"Динамика СФП существенно отличается от ФП. Общий объём СФП невелик (в среднем "
    f"{_sfp_totals.mean():.0f} событий в месяц), однако наблюдается аномальный всплеск "
    f"в {_sfp_peak_month} ({_fmt(_sfp_peak_val)} событий). "
    "Такие резкие скачки характерны для единовременных пакетных решений "
    "(например, массовое присвоение СФП по результатам квартальной переоценки портфеля). "
    "В отличие от ФП, устойчивой сезонности в СФП не прослеживается.")

# ═══ 14. APPROVED_SUM ═══
_heading(doc, "14. Анализ сумм по договорам (APPROVED_SUM)")

_rub_n = len(df_rub)
_rub_pct = _rub_n / len(df) * 100
_para(doc,
    f"Анализ проведён по рублёвым договорам (CURCY_CD = RUB): {_fmt(_rub_n)} записей "
    f"из {_fmt(len(df))} ({_rub_pct:.1f}% от общего числа). "
    "Для каждого уникального ИНН суммируется APPROVED_SUM по всем уникальным договорам "
    "(AGREEMENT_NUM). Результат распределяется по 8 корзинам в разрезе по сегментам.")

_para(doc, f"Уникальных пар ИНН + договор: {_fmt(len(df_agr))}. "
     f"Уникальных ИНН с суммой: {_fmt(len(inn_sum))} "
     f"({len(inn_sum)/_total_inn*100:.1f}% от {_fmt(_total_inn)} ИНН в анализе).")

doc.add_paragraph("Распределение ИНН по корзинам APPROVED_SUM (кол-во / % от сегмента):",
                   style="List Bullet").runs[0].bold = True
_merged_rows = []
for bkt in BUCKET_LABELS:
    row_vals = [bkt]
    for s in seg_order + ["Итого"]:
        abs_v = int(cross_abs.loc[bkt, s]) if bkt in cross_abs.index else 0
        pct_v = cross_pct.loc[bkt, s] if bkt in cross_pct.index else 0
        row_vals.append(f"{_fmt(abs_v)} ({pct_v:.1f}%)")
    _merged_rows.append(tuple(row_vals))
_tot_merged = ["ИТОГО"]
for s in seg_order + ["Итого"]:
    _tot_merged.append(_fmt(int(cross_abs.loc["Итого", s])))
_merged_rows.append(tuple(_tot_merged))
_mktable(doc, ["Корзина", "МкБ", "МСБ", "ККБ", "Итого"], _merged_rows,
         font_sz=8, num_cols={1,2,3,4}, col_widths=[3.5, 3, 3, 3, 3])

fig, ax = plt.subplots(figsize=(12, 5))
_pd = cross_abs.drop("Итого").drop(columns="Итого")
_pd.plot(kind="bar", ax=ax, edgecolor="white")
ax.set_title("Распределение ИНН по корзинам APPROVED_SUM (только RUB)", fontsize=13, fontweight="bold")
ax.set_xlabel("Корзина"); ax.set_ylabel("Количество ИНН"); ax.legend(title="Сегмент")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
_p4 = os.path.join(_TMP, "approved_sum.png"); fig.savefig(_p4, dpi=180); plt.close(fig)
_add_chart(doc, _p4)

doc.add_paragraph("Статистика APPROVED_SUM по сегментам (RUB):", style="List Bullet").runs[0].bold = True
rows_14c = []
for seg in seg_order:
    if seg in sum_stats.index:
        r = sum_stats.loc[seg]
        rows_14c.append((seg, _fmt(int(r["ИНН"])), str(r["Среднее"]), str(r["Медиана"]), str(r["Мин"]), str(r["Макс"])))
_mktable(doc, ["Сегмент","ИНН","Среднее (руб)","Медиана (руб)","Мин (руб)","Макс (руб)"],
         rows_14c, font_sz=8, num_cols={1,2,3,4,5}, col_widths=[2,2,3.5,3.5,3,3.5])

_bucket1_mkb = cross_pct.loc["до 100 млн", "МкБ"] if "до 100 млн" in cross_pct.index else 0
_bucket1_msb = cross_pct.loc["до 100 млн", "МСБ"] if "до 100 млн" in cross_pct.index else 0
_bucket1_kkb = cross_pct.loc["до 100 млн", "ККБ"] if "до 100 млн" in cross_pct.index else 0
_bucket1_tot = cross_pct.loc["до 100 млн", "Итого"] if "до 100 млн" in cross_pct.index else 0
_para(doc,
    f"Подавляющее большинство ИНН ({_bucket1_tot:.1f}%) сконцентрировано в корзине «до 100 млн руб.». "
    f"Наибольшая концентрация — в МкБ ({_bucket1_mkb:.1f}%) и МСБ ({_bucket1_msb:.1f}%). "
    f"ККБ демонстрирует наиболее равномерное распределение: лишь {_bucket1_kkb:.1f}% ИНН "
    "в первой корзине, при значительной доле крупных сделок (свыше 1 млрд руб.). "
    "Это отражает фундаментальное различие в масштабе кредитования между сегментами.")

# ═══ Заключение ═══
doc.add_page_break()
_heading(doc, "Заключение", level=1)
_para(doc,
    f"По результатам анализа CRM за период {DATE_FROM:%Y}–{DATE_TO:%Y} выявлено {_fmt(_te)} событий "
    f"по факторам проблемности у {_fmt(_total_inn)} уникальных ИНН в трёх сегментах бизнеса. "
    "Основные выводы:")
_mkb_inn = df[df["segment"]=="МкБ"]["inn"].nunique()
_msb_avg = (df[df["segment"]=="МСБ"]["TYPE_FP"]=="ФП").sum() / max(df[df["segment"]=="МСБ"]["inn"].nunique(), 1)
_sfp_pct = _se / _te * 100

conclusions = [
    f"За период {DATE_FROM:%Y}–{DATE_TO:%Y} зафиксировано {_fmt(_te)} событий по факторам проблемности "
    f"у {_fmt(_total_inn)} уникальных ИНН. Факторы проблемности (ФП) составляют {_fe/_te*100:.1f}% событий "
    f"({_fmt(_fe)}), существенные факторы проблемности (СФП) — {_se/_te*100:.1f}% ({_fmt(_se)}).",
    f"Микробизнес (МкБ) формирует основной массив заёмщиков — {_fmt(_mkb_inn)} из {_fmt(_total_inn)} "
    f"ИНН ({_mkb_inn/_total_inn*100:.1f}%), однако имеет наименьшую среднюю нагрузку "
    f"({_seg_stats['МкБ']['avg']:.2f} событий на ИНН). Наибольшая нагрузка — в МСБ "
    f"({_seg_stats['МСБ']['avg']:.2f} событий на ИНН).",
    f"Высокая концентрация ФП: топ-5 факторов покрывают {_top5_fp_sum/_fe*100:.1f}% всех событий ФП. "
    f"Лидер — ФП {_t20fp.index[0]} ({_t20fp.iloc[0]/_fe*100:.1f}%). "
    f"Среди СФП концентрация ещё выше: топ-3 покрывают {_top3_sfp_sum/_se*100:.1f}% событий.",
    "Лидирующие факторы существенно различаются по сегментам: в МкБ — нарушение обязательств "
    "по поддержанию оборотов, в МСБ — изменение финансовых показателей, "
    "в ККБ — сигналы внутренней рейтинговой системы. Это подтверждает необходимость "
    "посегментного подхода к мониторингу и управлению кредитным качеством.",
    f"Динамика ФП демонстрирует выраженную сезонность с пиками в апреле–мае и августе. "
    f"Максимум зафиксирован в {_fp_peak_month} ({_fmt(_fp_peak_val)} событий). "
    f"Динамика СФП менее стабильна, с аномальным всплеском в {_sfp_peak_month} ({_fmt(_sfp_peak_val)} событий).",
    f"Рублёвые договоры составляют {_rub_pct:.1f}% от выборки ({_fmt(_rub_n)} записей). "
    f"У {_fmt(len(inn_sum))} ИНН ({len(inn_sum)/_total_inn*100:.1f}%) заполнена сумма по договорам. "
    f"В МкБ и МСБ свыше {min(_bucket1_mkb, _bucket1_msb):.0f}% ИНН сконцентрировано "
    "в корзине «до 100 млн руб.»; в ККБ распределение значительно равномернее "
    "с существенной долей крупных сделок.",
]
for c in conclusions:
    doc.add_paragraph(c, style="List Bullet")

_para(doc, "Полные данные (все таблицы без ограничений по Top-N) доступны в файле приложения: "
     'Приложение к отчету "Анализ динамики ФП по сегментам бизнеса".xlsx.')

doc.save(_OUT)
print(f"Отчёт сохранён: {_OUT}")